Ensure the basic catalog structure is in place

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS league_pipeline;
CREATE SCHEMA IF NOT EXISTS league_pipeline.landing_zone;
CREATE SCHEMA IF NOT EXISTS league_pipeline.raw;
CREATE SCHEMA IF NOT EXISTS league_pipeline.bronze;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.players;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.masteries;
CREATE VOLUME IF NOT EXISTS league_pipeline.landing_zone.checkpoints;

Infer Schemas before-hand (could be run once a day to optimize performance, currently always runs since its very cheap)

In [ ]:
import json

for subdir in ["players", "masteries"]:
    schema_path = f"/Volumes/league_pipeline/landing_zone/checkpoints/{subdir}/{subdir}_schema.json"
    path = f"/Volumes/league_pipeline/landing_zone/{subdir}/"
    files = dbutils.fs.ls(path)
    new_schema_json = None

    if len(files) > 0:
        new_schema_json = spark.read.format("parquet").load(path).schema.json()
        
        dbutils.fs.put(schema_path, new_schema_json, overwrite=True)
        print(f"Schema refreshed from {path}")
    else:
        print(f"No files in landing zone for {subdir} — reusing existing schema.")

    if new_schema_json:
        print(new_schema_json)

Auto Loader Job

In [0]:
import json
from pyspark.sql.types import StructType
from pyspark.sql.functions import col, current_timestamp

for subdir in ["players", "masteries"]:
    schema_path = f"/Volumes/league_pipeline/landing_zone/checkpoints/{subdir}/{subdir}_schema.json"
    landing_path = f"dbfs:/Volumes/league_pipeline/landing_zone/{subdir}"
    raw_table = f"league_pipeline.raw.{subdir}"

    try:
        landing_files = dbutils.fs.ls(landing_path)
    except Exception as e:
        print(f"Error accessing landing path {landing_path} for {subdir}: {e}")
        continue

    if not landing_files:
        print(f"No files in landing zone for {subdir}; skipping ingestion.")
        continue

    table_exists = spark.catalog.tableExists(raw_table)
    print(f"Raw table {raw_table} found: {table_exists}")

    # schema check
    try:
        dbutils.fs.ls(schema_path)
        schema_exists = True
    except Exception:
        schema_exists = False

    if schema_exists:
        with open(schema_path) as f:
            schema_struct = StructType.fromJson(json.load(f))
    else:
        schema_struct = spark.read.format("parquet").load(landing_path).schema
        dbutils.fs.put(schema_path, schema_struct.json(), overwrite=True)
        print(f"Created schema metadata for {subdir} from landing files.")

    checkpoint_schema = f"/Volumes/league_pipeline/landing_zone/checkpoints/{subdir}/schema"
    checkpoint_data = f"/Volumes/league_pipeline/landing_zone/checkpoints/{subdir}/data"

    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "parquet")
        .option("cloudFiles.schemaLocation", checkpoint_schema)
        .schema(schema_struct)
        .load(landing_path)
        .withColumn("_ingested_at", current_timestamp())
    )

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_data)
        .trigger(availableNow=True)
        .toTable(raw_table)
    )
    query.awaitTermination()

    print(f"Streaming ingestion completed for {raw_table}.")